In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    to_date,
    to_timestamp,
    when,
    current_timestamp,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    IntegerType
)


# ============================================
# STORAGE PATHS
# ============================================

bronze_root = (
    "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/"
)

silver_root = (
    "abfss://silver@bankingdelakevishal.dfs.core.windows.net/"
)

In [0]:
# ============================================
# CONFIGURATION & SCHEMA SETUP
# ============================================
CATALOG_NAME = "banking_lakehouse_db2"
SCHEMA_NAME = "silver"
TABLE_NAME = "customer"
FULL_TABLE_NAME = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{TABLE_NAME}"

bronze_root = "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/"
silver_root = "abfss://silver@bankingdelakevishal.dfs.core.windows.net/"

# Ensure Silver schema exists inside your catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

# ============================================
# READ BRONZE CUSTOMER DATA
# ============================================
customer_bronze = spark.read.parquet(f"{bronze_root}customer/")
bronze_count = customer_bronze.count()
print(f"Bronze Record Count: {bronze_count}")

# ============================================
# CLEAN AND TRANSFORM
# ============================================
customer_cleaned = (
    customer_bronze
    .select(
        trim(col("customer_id")).alias("customer_id"),
        trim(col("first_name")).alias("first_name"),
        trim(col("last_name")).alias("last_name"),
        upper(trim(col("gender"))).alias("gender"),
        to_date(trim(col("date_of_birth")), "yyyy-MM-dd").alias("date_of_birth"),
        lower(trim(col("email"))).alias("email"),
        trim(col("phone")).alias("phone"),
        upper(trim(col("pan_number"))).alias("pan_number"),
        trim(col("aadhaar_number")).alias("aadhaar_number"),
        trim(col("occupation")).alias("occupation"),
        trim(col("annual_income")).cast("decimal(18,2)").alias("annual_income"),
        upper(trim(col("marital_status"))).alias("marital_status"),
        to_date(trim(col("customer_since")), "yyyy-MM-dd").alias("customer_since"),
        upper(trim(col("kyc_status"))).alias("kyc_status"),
        upper(trim(col("risk_category"))).alias("risk_category"),
        trim(col("branch_id")).alias("branch_id"),
        trim(col("city")).alias("city"),
        trim(col("state")).alias("state"),
        upper(trim(col("country"))).alias("country"),
        upper(trim(col("customer_status"))).alias("customer_status")
    )
)

# ============================================
# REMOVE INVALID RECORDS
# ============================================
customer_valid = (
    customer_cleaned
    # Customer ID validation
    .filter(col("customer_id").isNotNull())
    .filter(trim(col("customer_id")) != "")
    
    # Email validation
    .filter(col("email").isNotNull())
    .filter(trim(col("email")) != "")
    .filter(col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"))
    
    # Date of birth validation
    .filter(col("date_of_birth").isNotNull())
    .filter(col("date_of_birth") <= current_timestamp())
    
    # Safety deduplication
    .dropDuplicates(["customer_id"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

# ============================================
# WRITE FRESH DATA (OVERWRITE PATH & TABLE)
# ============================================
# 1. Overwrite raw Delta files in ADLS
(
    customer_valid
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{silver_root}customer/")
)

# 2. Overwrite / Register managed Unity Catalog table
(
    customer_valid
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

# ============================================
# VALIDATION METRICS
# ============================================
valid_count = customer_valid.count()
print(f"Silver Valid Record Count: {valid_count}")
print(f"Dropped Records: {bronze_count - valid_count}")

customer_valid.printSchema()
display(customer_valid.limit(10))